# Agents

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import pathlib

import jinja2
from IPython.display import Code

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets, types
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.tools import lint
from hslu.dlm03.util import file as file_lib

# Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
code_folder = pathlib.Path("/Users/vincent/Development/Test/")

# Tool Definitions

In [ ]:
Code("../util/file.py")

In [ ]:
def list_files(path: str, glob: str) -> list[str]:
    return [str(file) for file in pathlib.Path(path).glob(glob)]

list_files_tool = {
    "type": "function",
    "function": {
        "name": "list_files",
        "description": "List the files from the given path using the given glob expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                },
                "glob": {
                    "type": "string",
                },
            },
            "required": [
                "path",
                "glob",
            ],
        },
        "strict": True,
    },
}

In [ ]:
def read_file(path: str) -> file_lib.File | str:
    file_path = pathlib.Path(path)
    if not file_path.exists():
        return "File not found."
    if file_path.is_dir():
        return "Given path is a directory."
    if file_path.is_file():
        return file_lib.File.read_from(pathlib.Path(path))
    return "File could not be read."

read_file_tool = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Reads the file content for the given path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                },
            },
            "required": [
                "path",
            ],
        },
        "strict": True,
    },
}

In [ ]:
def lint_file(path: str) -> lint.Issues:
    return lint.lint(pathlib.Path(path))

lint_file_tool = {
    "type": "function",
    "function": {
        "name": "lint_file",
        "description": "Runs the Python linter on the given file path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                },
            },
            "required": [
                "path",
            ],
        },
        "strict": True,
    },
}


In [ ]:
def write_file(path: str, content: str) -> str:
    file_path = pathlib.Path(path)
    try:
        file_path.write_text(content)
    except Exception as e:
        return f"Failed to write file: {e}"
    else:
        return f"Successfully wrote {file_path}"

write_file_tool = {
    "type": "function",
    "function": {
        "name": "write_file",
        "description": "Runs the Python linter on the given file path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                },
                "content": {
                    "type": "string",
                },
            },
            "required": [
                "path",
                "content",
            ],
        },
        "strict": True,
    },
}


# Agentic Harness

In [ ]:
def execute(tool_call: types.ToolCall) -> types.ToolCallOutput:
    tool_name = tool_call.function.name
    tool_arguments = json.loads(tool_call.function.arguments)
    call_id = tool_call.id
    match tool_name:
        case "list_files":
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "content": str(list_files(**tool_arguments)),
            }
        case "read_file":
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "content": str(read_file(**tool_arguments)),
            }
        case "lint_file":
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "content": str(lint_file(**tool_arguments)),
            }
        case "write_file":
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "content": str(write_file(**tool_arguments)),
            }
        case _:
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "content": f"Unknown tool {tool_name}",
            }


In [ ]:
system_instructions_template_string = \
"""You are an expert Software Engineer tasked to solve issues found in code.
You will be provided with some code files, and a list of issues found by a linter,
and you should output the fixed code such that the issues are not present anymore."""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please fix the files in {{ code_folder }}."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message = user_message_template.render(code_folder=code_folder)
system_instructions = system_instructions_template.render()

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
done = False
while not done:
    response = backend.generate(
        chat,
        tools=[list_files_tool, read_file_tool, lint_file_tool, write_file_tool],
    )
    done = True
    message = response.choices[0].message
    chat.append(message)
    if message.tool_calls:
        for tool_call in message.tool_calls:
            done = False
            tool_call_result = execute(tool_call)
            chat.append(tool_call_result)